In [1]:
import os
import glob
from pathlib import Path

current_dir = Path.cwd()
data_dir = current_dir.parent/'data'
processed_dir = current_dir.parent/'processed_data'
os.makedirs(processed_dir, exist_ok=True)
supporting_dir = current_dir.parent/'supporting_data'
viz_dir = current_dir.parent/'viz'
os.makedirs(viz_dir, exist_ok=True)

In [2]:
import pandas as pd
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 25)
import numpy as np
import re
import json
import matplotlib.pyplot as plt
import seaborn as sns

import geopandas as gpd
from shapely.geometry import Point, shape
import folium
from folium.utilities import JsCode
from folium.features import GeoJsonPopup
from folium.plugins.timeline import Timeline, TimelineSlider

In [3]:
gdf_list = []

for file in glob.glob(f'{data_dir}/*Gang*.geojson'):
    file_year = ''.join(re.findall(r'\d+', str(file)))
    gdf= gpd.read_file(file)
    gdf['map_year'] = file_year
    gdf.columns = gdf.columns.str.replace('__', '_')
    gdf_list.append(gdf)

raw_gdf= pd.concat(gdf_list)
raw_gdf.columns= raw_gdf.columns.str.lower()

raw_gdf = raw_gdf.map(lambda x: x.lower() if isinstance(x, str) else x)
raw_gdf['map_year'] = raw_gdf['map_year'].astype(int)

raw_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1052 entries, 0 to 50
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   objectid      862 non-null    float64 
 1   gang_name     1050 non-null   object  
 2   shape_area    732 non-null    float64 
 3   shape_length  638 non-null    float64 
 4   geometry      988 non-null    geometry
 5   map_year      1052 non-null   int64   
 6   shape_area    350 non-null    float64 
 7   shape_length  350 non-null    float64 
 8   fid           190 non-null    float64 
 9   shape_area    96 non-null     float64 
 10  shape_len     96 non-null     float64 
 11  shape_leng    94 non-null     float64 
dtypes: float64(9), geometry(1), int64(1), object(1)
memory usage: 106.8+ KB


In [4]:
# Find all shape_area columns
shape_area = raw_gdf.loc[:, raw_gdf.columns == 'shape_area']

# Combine them, taking the first non-null value in each row
raw_gdf['shape_area_combined'] = shape_area.bfill(axis=1).iloc[:, 0]

# Drop the duplicate shape_area columns
raw_gdf = raw_gdf.drop(columns='shape_area')

# Rename the combined column
raw_gdf = raw_gdf.rename(columns={'shape_area_combined': 'shape_area'})

In [5]:
raw_gdf = raw_gdf.to_crs(epsg=3435)

# Calculate shapre_area in square mile
raw_gdf['shape_area_sqm'] = raw_gdf.geometry.area / (5280 ** 2)

In [6]:
allGangs_totalShapeArea = raw_gdf.groupby('map_year')['shape_area_sqm'].sum()

allGangs_totalShapeArea

map_year
2007    71.172256
2008    72.216419
2009    71.981040
2010    80.561723
2011    79.855857
2012    62.037174
2014    65.260860
2015    66.622091
2016    65.700803
2017    64.603170
2018    65.062397
2019    63.705038
2020    64.040688
2021    64.776293
2022    64.399863
2023    64.509211
2024    62.904418
2025    63.022023
Name: shape_area_sqm, dtype: float64

In [7]:
allGangs_totalShapeArea = raw_gdf.groupby('map_year')['shape_area_sqm'].sum()

# In 2025, the city of Chicago has 3.6 square miles less in gang territority than in 2015
allGangs_totalShapeArea.loc[2025] - allGangs_totalShapeArea.loc[2015]

np.float64(-3.6000676311720596)

In [38]:
# Clean up mispelling in the gang_name field
gang_named_cleanup_dict = {
    '12th st players': '12th street players',
    'black p stone': 'black p stones',
    'krazy get down boys': 'krazy getdown boys',
    'two six': 'two-six',
    'ylo disciples': 'young latin organization disciples',
    'young latin organization disciple': 'young latin organization disciples',
    'ylo cobras': 'young latin organization cobras'
}

raw_gdf['gang_name_cleaned'] = (
    raw_gdf['gang_name']
    .str.strip()
    .replace(gang_named_cleanup_dict)
    .fillna(raw_gdf['gang_name'])
)

In [39]:
gangs_summary = raw_gdf.pivot_table(
    index='map_year',
    columns='gang_name_cleaned',
    values='shape_area_sqm',
    aggfunc='sum'
)\
    .fillna(0)\
    .T

gangs_summary = gangs_summary.reset_index()\
                .sort_values('gang_name_cleaned', ascending=True)\
                .set_index('gang_name_cleaned')
gangs_summary

map_year,2007,2008,2009,2010,2011,2012,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
gang_name_cleaned,,,,,,,,,,,,,,,,,,
12th street players,0.001983,0.001983,0.001983,0.000000,0.000000,0.009139,0.009078,0.008977,0.008977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4ch bps cvl,0.000000,0.069998,0.069998,0.204329,0.204329,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ambrose,0.386158,0.386158,0.386158,0.215258,0.215258,0.439129,0.438329,0.438329,0.438329,0.454350,0.454350,0.454350,0.454350,0.454350,0.454350,0.454350,0.454350,0.454350
ashland vikings,0.055037,0.055037,0.055037,0.061817,0.061817,0.043552,0.036055,0.024126,0.009905,0.009905,0.009905,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
aztecas,0.000000,0.000000,0.000000,0.016143,0.016143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
undertaker vice lords,0.154586,0.154586,0.154586,0.080975,0.080975,0.182729,0.193173,0.085796,0.085796,0.117699,0.221571,0.221571,0.200697,0.200697,0.026657,0.200697,0.171968,0.171968
unknown vice lords,0.447175,0.447175,0.447175,0.255697,0.255697,0.568256,0.598409,0.608932,0.608932,0.685227,0.662960,0.662960,0.773071,0.773071,0.636459,0.717085,0.753628,0.753628
vice lords,0.000000,0.000000,0.000000,0.000000,0.000000,0.178528,0.220118,0.214369,0.214369,0.210899,0.185291,0.185291,0.185291,0.135900,0.135900,0.054854,0.054854,0.054854


In [53]:
mid2010s_mean = (
    gangs_summary[[2014, 2015]].mean(axis=1)

    )

current_mean = (
    gangs_summary.iloc[:, -2:].mean(axis=1)
    )

# Gangs that have lost and gained terrority (in square miles) in the last decade
gang_turfChanged= (current_mean - mid2010s_mean).sort_values(ascending=False)\
    .reset_index(name='turf_change_sqm')

pd.concat([gang_turfChanged.nlargest(10, 'turf_change_sqm'), gang_turfChanged.nsmallest(5, 'turf_change_sqm')])

,gang_name_cleaned,turf_change_sqm
0,satan disciples,0.328124
1,new breed,0.298115
2,black souls,0.258093
3,latin brothers organization,0.255341
4,krazy getdown boys,0.255316
5,latin dragons,0.196803
6,black disciples,0.152183
7,unknown vice lords,0.149958
8,maniac latin disciples,0.144921
9,latin eagles,0.079045


In [43]:
chiBoundaries_gdf = gpd.read_file(f'{data_dir}/Boundaries_-_Community_Areas_20261015.geojson')
chiBoundaries_gdf= chiBoundaries_gdf.iloc[:, -6:]

chiBoundaries_gdf = chiBoundaries_gdf.set_crs(
    'EPSG:4326',
    allow_override=True
)

chiBoundaries_gdf = chiBoundaries_gdf.to_crs(raw_gdf.crs)

chiBoundaries_gdf.crs

<Projected CRS: EPSG:3435>
Name: NAD83 / Illinois East (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - Illinois - counties of Boone; Champaign; Clark; Clay; Coles; Cook; Crawford; Cumberland; De Kalb; De Witt; Douglas; Du Page; Edgar; Edwards; Effingham; Fayette; Ford; Franklin; Gallatin; Grundy; Hamilton; Hardin; Iroquois; Jasper; Jefferson; Johnson; Kane; Kankakee; Kendall; La Salle; Lake; Lawrence; Livingston; Macon; Marion; Massac; McHenry; McLean; Moultrie; Piatt; Pope; Richland; Saline; Shelby; Vermilion; Wabash; Wayne; White; Will; Williamson.
- bounds: (-89.27, 37.06, -87.02, 42.5)
Coordinate Operation:
- name: SPCS83 Illinois East zone (US survey foot)
- method: Transverse Mercator
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [44]:
raw_gdf_copy = raw_gdf.explode(index_parts=False)\
                      .reset_index(drop=True)\
                      .to_crs(chiBoundaries_gdf.crs)

raw_gdf_copy = gpd.sjoin(
    raw_gdf_copy,
    chiBoundaries_gdf[['community', 'geometry']], 
    how='left',
    predicate='within'
)

raw_gdf_copy = raw_gdf_copy.rename(columns = {'community_right': 'community'})
raw_gdf_copy['shape_area_sqm_cleaned'] = raw_gdf_copy.geometry.area / (5280 ** 2) 

raw_gdf_copy.head().T

,0,1,2,3,4
objectid,1.0,1.0,1.0,1.0,2.0
gang_name,ambrose,ambrose,ambrose,ambrose,ashland vikings
shape_length,8395.432312,8395.432312,8395.432312,8395.432312,3383.750569
map_year,2011,2011,2011,2011,2011
shape_length,NaN,NaN,NaN,NaN,NaN
fid,NaN,NaN,NaN,NaN,NaN
shape_len,NaN,NaN,NaN,NaN,NaN
shape_leng,NaN,NaN,NaN,NaN,NaN
shape_area,1005116.414062,1005116.414062,1005116.414062,1005116.414062,289840.550781
shape_area_sqm,0.215258,0.215258,0.215258,0.215258,0.061817


In [45]:
community_summary = raw_gdf_copy.groupby(['map_year', 'community'])\
            .shape_area_sqm_cleaned.sum()\
            .unstack()\
            .fillna(0)

community_summary.tail().T

map_year,2021,2022,2023,2024,2025
community,,,,,
ALBANY PARK,0.126587,0.126587,0.094976,0.126587,0.126587
ARCHER HEIGHTS,0.000000,0.000000,0.000000,0.000000,0.000000
ARMOUR SQUARE,0.031778,0.031778,0.000000,0.031778,0.031778
ASHBURN,1.187132,1.187132,0.000000,1.187132,1.187132
AUBURN GRESHAM,0.513534,0.513534,0.492151,0.417502,0.478234
...,...,...,...,...,...
WEST LAWN,0.000000,0.000000,0.000000,0.000000,0.000000
WEST PULLMAN,0.628487,0.628487,0.628486,0.628487,0.628487
WEST RIDGE,0.632867,0.632867,0.632866,0.632867,0.632867


In [46]:
# Community area that saw less gang turf in the last decade
(
community_summary[community_summary.index.isin([2014, 2015])].mean(axis=0) - 
 community_summary[community_summary.index.isin([2024, 2025])].mean(axis=0)
 )\
    .sort_values(ascending=False)\
    .head(10)

community
AVONDALE              0.630324
PORTAGE PARK          0.542788
HUMBOLDT PARK         0.438185
ENGLEWOOD             0.326230
ROGERS PARK           0.318364
NORTH CENTER          0.303604
EAST GARFIELD PARK    0.247276
WOODLAWN              0.238495
LOGAN SQUARE          0.225735
EAST SIDE             0.197328
dtype: float64

In [48]:
# Find gangs in 2025 made up 80 percent of the gang turf 
top_gangs_2025 = (
    raw_gdf[raw_gdf.map_year == 2025]
    .groupby('gang_name_cleaned')
    .shape_area.sum()
    .div(raw_gdf[raw_gdf.map_year == 2025].shape_area.sum())
    .sort_values(ascending=False)
    .cumsum()
    .loc[lambda x: x <= 0.8]
)

top_gangs_2025_list = top_gangs_2025.index.to_list()
top_gangs_2025_gdf = raw_gdf[raw_gdf.gang_name.isin(top_gangs_2025_list)]

In [49]:
import folium
from folium.plugins import Timeline, TimelineSlider
from folium.features import GeoJsonPopup
from folium.utilities import JsCode


gdf = top_gangs_2025_gdf[
    ["gang_name_cleaned", "geometry", "map_year"]
].copy()

gdf = gdf.dropna(
    subset=["map_year", "gang_name_cleaned"]
).copy()

gdf["map_year"] = gdf["map_year"].astype(int)

print("Years:", sorted(gdf["map_year"].unique()))
print("Gangs:", sorted(gdf["gang_name_cleaned"].unique()))



gdf = gdf.to_crs(epsg=4326)


representative_points = gdf.geometry.representative_point()

center_lat = representative_points.y.mean()
center_lon = representative_points.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles=None
)


folium.TileLayer(
    tiles=(
        "https://basemaps.cartocdn.com/rastertiles/light_all/"
        "{z}/{x}/{y}.png"
        "?key=cb1_3t1q_1_0960046504f90cb9125e4501"
    ),
    attr="© OpenStreetMap contributors © CARTO",
    name="Chicago Gangland",
    overlay=False,
    control=True,
    show=True
).add_to(m)


gang_names = sorted(gdf["gang_name_cleaned"].unique())

colors = [
    "#e41a1c",
    "#377eb8",
    "#4daf4a",
    "#984ea3",
    "#ff7f00",
    "#a65628",
    "#f781bf",
    "#999999",
    "#66c2a5",
    "#fc8d62",
    "#8da0cb",
    "#e78ac3",
    "#a6d854",
    "#ffd92f",
    "#e5c494",
    "#b3b3b3",
    "#1b9e77",
    "#d95f02",
    "#7570b3",
    "#e7298a"
]

gang_colors = {
    gang: colors[i % len(colors)]
    for i, gang in enumerate(gang_names)
}


timelines = []

for gang in gang_names:

    gang_gdf = gdf[
        gdf["gang_name_cleaned"] == gang
    ].copy()

    gang_geojson = gang_gdf.__geo_interface__

    # Add timeline dates
    for feature in gang_geojson["features"]:

        year = feature["properties"]["map_year"]

        feature["properties"]["start"] = f"{year}-01-01"
        feature["properties"]["end"] = f"{year}-12-31"

    gang_color = gang_colors[gang]

    # Timeline styling
    timeline_style = JsCode(
        f"""
        function(data) {{
            return {{
                stroke: true,
                color: "{gang_color}",
                weight: 2,
                fillColor: "{gang_color}",
                fillOpacity: 0.50
            }};
        }}
        """
    )

    # Gang layer
    gang_group = folium.FeatureGroup(
        name=str(gang),
        overlay=True,
        show=True
    )

    gang_group.add_to(m)

    # Timeline
    timeline = Timeline(
        gang_geojson,
        name=str(gang),
        style=timeline_style,
        overlay=True,
        control=False,
        show=True
    )

    timeline.add_to(gang_group)

    # Popup
    GeoJsonPopup(
        fields=["gang_name_cleaned", "map_year"],
        aliases=["Gang:", "Year:"],
        labels=True,
        sticky=False
    ).add_to(timeline)

    timelines.append(timeline)


TimelineSlider(
    auto_play=False,
    show_ticks=True,
    enable_keyboard_controls=True,
    playback_duration=15000,
    date_options="YYYY"
).add_timelines(*timelines).add_to(m)


folium.LayerControl(
    collapsed=False
).add_to(m)

layer_control_css = """
<style>

.leaflet-control-layers {
    font-size: 16px !important;
    line-height: 1.6 !important;
    min-width: 280px !important;
    max-height: 600px !important;
    overflow-y: auto !important;
}

.leaflet-control-layers-list {
    padding: 8px 12px !important;
}

.leaflet-control-layers-overlays label,
.leaflet-control-layers-base label {
    margin-bottom: 6px !important;
}

.leaflet-control-layers-selector {
    width: 18px !important;
    height: 18px !important;
    margin-right: 8px !important;
}

.gang-layer-button {
    display: inline-block;
    width: 48%;
    margin-top: 10px;
    padding: 8px 5px;
    background-color: white;
    border: 1px solid #999;
    border-radius: 4px;
    font-size: 14px;
    font-weight: bold;
    cursor: pointer;
    box-sizing: border-box;
}

.gang-layer-button:hover {
    background-color: #f0f0f0;
}

.gang-layer-button + .gang-layer-button {
    margin-left: 2%;
}

</style>
"""

m.get_root().html.add_child(
    folium.Element(layer_control_css)
)

select_deselect_js = """
<script>

function setAllGangs(selected) {

    var layerControl = document.querySelector(
        '.leaflet-control-layers-overlays'
    );

    if (!layerControl) {
        return;
    }

    var checkboxes = layerControl.querySelectorAll(
        'input[type="checkbox"]'
    );

    checkboxes.forEach(function(checkbox) {

        if (checkbox.checked !== selected) {
            checkbox.click();
        }

    });
}

function selectAllGangs() {
    setAllGangs(true);
}

function deselectAllGangs() {
    setAllGangs(false);
}

</script>
"""

m.get_root().html.add_child(
    folium.Element(select_deselect_js)
)


button_html = """
<script>

document.addEventListener("DOMContentLoaded", function() {

    var interval = setInterval(function() {

        var overlayContainer = document.querySelector(
            '.leaflet-control-layers-overlays'
        );

        if (!overlayContainer) {
            return;
        }

        // Prevent duplicate buttons
        if (document.getElementById("gang-layer-buttons")) {
            clearInterval(interval);
            return;
        }

        // Button container
        var buttonContainer = document.createElement("div");

        buttonContainer.id = "gang-layer-buttons";
        buttonContainer.style.width = "100%";
        buttonContainer.style.display = "flex";
        buttonContainer.style.marginTop = "5px";

        // Select All button
        var selectButton = document.createElement("button");

        selectButton.id = "select-all-gangs";
        selectButton.type = "button";
        selectButton.className = "gang-layer-button";
        selectButton.innerHTML = "Select All Gangs";

        selectButton.onclick = function() {
            selectAllGangs();
        };

        // Deselect All button
        var deselectButton = document.createElement("button");

        deselectButton.id = "deselect-all-gangs";
        deselectButton.type = "button";
        deselectButton.className = "gang-layer-button";
        deselectButton.innerHTML = "Deselect All Gangs";

        deselectButton.onclick = function() {
            deselectAllGangs();
        };

        // Add buttons
        buttonContainer.appendChild(selectButton);
        buttonContainer.appendChild(deselectButton);

        overlayContainer.appendChild(buttonContainer);

        clearInterval(interval);

    }, 100);

});

</script>
"""

m.get_root().html.add_child(
    folium.Element(button_html)
)

filename = "top_gangs"

output_path = f"{viz_dir}/{filename}.html"

m.save(output_path)

Years: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Gangs: ['black disciples', 'black p stones', 'four corner hustlers', 'gangster disciples', 'latin kings', 'maniac latin disciples', 'satan disciples', 'traveling vice lords', 'two-six']
